Imports

In [ ]:
import random
import itertools
from math import factorial

Order Crossover Function

In [ ]:
def order_crossover(parent_1, parent_2):
  J = len(parent_1)
  c_1 = random.randint(0, J - 2)
  c_2 = random.randint(c_1 + 1, J - 1)

  def make_child(pA, pB):
    child = [None] * J
    keep = set()

    # Copy interval from pA
    for i in range(c_1, c_2 + 1):
      child[i] = pA[i]
      keep.add(pA[i])

    write = (c_2 + 1) % J
    read = (c_2 + 1) % J

    while None in child:
      gene = pB[read]
      if gene not in keep:
        while child[write] is not None:
          write = (write + 1) % J
        child[write] = gene
        write = (write + 1) % J
      read = (read + 1) % J

    return child
  child_1 = make_child(parent_1, parent_2)
  child_2 = make_child(parent_2, parent_1)
  return child_1, child_2

First Ready Job Function

In [ ]:
# Scan perm, returns first job with remaining ops whose ready time <= t
def first_ready_job(perm, job_ready, job_next_op, t, num_ops_per_job):
  for j in perm:
    if job_next_op[j] < num_ops_per_job and job_ready[j] <= t:
      return j
  return None


Next Job Ready Time Function

In [ ]:
# Returns earliest time when any unfinished job becomes ready
def next_job_ready_time(job_ready, job_next_op, num_ops_per_job):
  t_min = float('inf')
  for j in range(len(job_ready)):
    if job_next_op[j] < num_ops_per_job:
      t_min = min(t_min, job_ready[j])
  return t_min

Makespan Function

In [ ]:
def makespan(perm, proc_time, M):
  num_jobs = len(proc_time)
  num_ops_per_job = len(proc_time[0])

  machine_free = [0] * M
  job_ready = [0] * num_jobs
  job_next_op = [0] * num_jobs
  remaining = num_jobs * num_ops_per_job

  while remaining > 0:
    m = min(range(M), key=lambda i: machine_free[i])
    t = machine_free[m]

    j = first_ready_job(perm, job_ready, job_next_op, t, num_ops_per_job)

    if j is None:
      t = next_job_ready_time(job_ready, job_next_op, num_ops_per_job)
      j = first_ready_job(perm, job_ready, job_next_op, t, num_ops_per_job)

    k = job_next_op[j]
    dur = proc_time[j][k]
    start = max(machine_free[m], job_ready[j])
    finish = start + dur

    machine_free[m] = finish
    job_ready[j] = finish
    job_next_op[j] = k + 1
    remaining -= 1

  return max(machine_free)

Round-Robin Makespan

In [ ]:
def makespan_round_robin(perm, proc_time, M):
    J = len(proc_time)
    N = len(proc_time[0])

    machine_free = [0] * M
    job_ready    = [0] * J
    job_next_op  = [0] * J

    op_counter = 0
    remaining  = J * N

    while remaining > 0:
        t = min(machine_free)

        # pick first ready job in perm; if none ready, jump to next job ready time
        j = first_ready_job(perm, job_ready, job_next_op, t, N)
        if j is None:
            t = next_job_ready_time(job_ready, job_next_op, N)
            j = first_ready_job(perm, job_ready, job_next_op, t, N)

        # assign this operation to machine in round-robin order
        m = op_counter % M
        k = job_next_op[j]
        dur = proc_time[j][k]

        start  = max(machine_free[m], job_ready[j])
        finish = start + dur

        machine_free[m] = finish
        job_ready[j]    = finish
        job_next_op[j]  = k + 1

        op_counter += 1
        remaining  -= 1

    return max(machine_free)


proc = [
    [3, 5],
    [2, 4],
]
perm = [0, 1]   # job order
M = 2

ms = makespan_round_robin(perm, proc, M)
print("Round-robin makespan (tiny test) =", ms)


Round-robin makespan (tiny test) = 8


Fitness Function

In [ ]:
def fitness(chrom, proc_time, M):
  ms = makespan(chrom, proc_time, M)
  return 1.0 / ms, ms

Fitness Round Robin


In [ ]:
def fitness_round_robin(chrom, proc_time, M):
    ms = makespan_round_robin(chrom, proc_time, M)
    return 1.0 / ms, ms

Mutate Swap Function

In [ ]:
# Swap two random positions with probability pm
def mutate_swap(chrom, pm=0.05):
  if random.random() < pm:
    i, j = random.sample(range(len(chrom)), 2)
    chrom[i], chrom[j] = chrom[j], chrom[i]
  return chrom

Population Initialization Function

In [ ]:
# Enumerate all permutations if J! <= pop_size; otherwise sample unique perms
def init_population(J, pop_size=120):
  total = factorial(J)
  if total <= pop_size:
    return [list(p) for p in itertools.permutations(range(J))]
  else:
    pop, seen = [], set()
    while len(pop) < pop_size:
      chrom = tuple(random.sample(range(J), J))
      if chrom not in seen:
        seen.add(chrom)
        pop.append(list(chrom))
    return pop

Select Roulette Function

In [ ]:
# Roulette wheel selection of k chromosomes
def select_roulette(pop, proc_time, M, K = 10, fitness_fn=fitness):
  fits = []
  for c in pop:
    f, _ = fitness_fn(c, proc_time, M)
    fits.append(f)
  s = sum(fits)
  probs = None if s == 0 else [f/s for f in fits]
  chosen = random.choices(population=pop, weights=probs, k=K)
  return [c[:] for c in chosen]

Main GA Loop Function

In [ ]:
def run_ga(proc_time, M, pop_size=120, K=10, gens=100, pm=0.05, seed=0, ms_fn=None):

  if ms_fn is None:
    ms_fn = makespan

  random.seed(seed)
  J = len(proc_time)

  def fitness_with(chrom):
    ms = ms_fn(chrom, proc_time, M)
    return 1.0 / ms, ms

  pop = init_population(J, pop_size=pop_size)
  best_chrom = None
  best_ms = float('inf')

  for g in range(gens):

    fits = [fitness_with(c)[0] for c in pop]
    s = sum(fits)
    probs = None if s == 0 else [f/s for f in fits]
    selected = random.choices(population=pop, weights=probs, k=K)
    selected = [c[:] for c in selected]

    children = []
    for i in range(K):
      for j in range(i+1, K):
        c_1, c_2 = order_crossover(selected[i], selected[j])
        children.append(mutate_swap(c_1[:], pm=pm))
        children.append(mutate_swap(c_2[:], pm=pm))

    for c in children:
      ms = ms_fn(c, proc_time, M)
      if ms < best_ms:
        best_ms = ms
        best_chrom = c[:]

    pop.extend(children)
    pop.sort(key=lambda c: ms_fn(c, proc_time, M))
    pop = pop[:pop_size]

  return best_chrom, best_ms


Random Job Generator

In [ ]:
# Generate J jobs with N ops each; durations
def rand_jobs(J, N, low=5, high=50, seed=0):
  random.seed(seed)
  return [[random.randint(low, high) for _ in range(N)] for _ in range(J)]

R3

In [ ]:
proc_time_R3 = [[3, 6], [10, 1], [3, 2], [2, 4], [8, 8]]
M_R3 = 2

best_chrom_R3, best_ms_R3 = run_ga(
    proc_time_R3, M_R3,
    pop_size=120, K=10, gens=100, pm=0.05, seed=42,
    ms_fn=makespan
)
print("R3 best chromosome (0-index): ", best_chrom_R3)
print("R3 best chromosome (1-index): ", [x+1 for x in best_chrom_R3])
print("R3 best makespan:", best_ms_R3)

R3 best chromosome (0-index):  [3, 0, 4, 2, 1]
R3 best chromosome (1-index):  [4, 1, 5, 3, 2]
R3 best makespan: 25


R4

In [ ]:
proc_time_R4 = rand_jobs(J=10, N=3, low=5, high=50, seed=7)
M_R4 = 5

best_chrom_R4, best_ms_R4 = run_ga(
    proc_time_R4, M_R4,
    pop_size=120, K=10, gens=100, pm=0.05, seed=7,
    ms_fn=makespan_round_robin
)

random.seed(7)
pop0 = init_population(10, pop_size=120)
selected0 = select_roulette(pop0, proc_time_R4, M_R4, K=10, fitness_fn=fitness_round_robin)
baseline0 = random.choice(selected0)
baseline_ms_R4 = makespan_round_robin(baseline0, proc_time_R4, M_R4)

print("R4 best makespan:", best_ms_R4)
print("R4 baseline (pre-variation) makespan:", baseline_ms_R4)
print("R4 improvement:", baseline_ms_R4 - best_ms_R4)

R4 best makespan: 157
R4 baseline (pre-variation) makespan: 180
R4 improvement: 23


R5

In [ ]:
proc_time_R5 = rand_jobs(J=10, N=5, low=5, high=50, seed=11)
M_R5 = 3

best_chrom_R5, best_ms_R5 = run_ga(
    proc_time_R5, M_R5,
    pop_size=120, K=10, gens=100, pm=0.05, seed=11,
    ms_fn=makespan_round_robin
)

random.seed(11)
pop0 = init_population(10, pop_size=120)
selected0 = select_roulette(pop0, proc_time_R5, M_R5, K=10, fitness_fn=fitness_round_robin)
baseline0 = random.choice(selected0)
baseline_ms_R5 = makespan_round_robin(baseline0, proc_time_R5, M_R5)

print("R5 best makespan:", best_ms_R5)
print("R5 baseline (pre-variation) makespan:", baseline_ms_R5)
print("R5 improvement:", baseline_ms_R5 - best_ms_R5)


R5 best makespan: 480
R5 baseline (pre-variation) makespan: 542
R5 improvement: 62
